In [ ]:
import numpy as np
import pandas as pd

FEATURES = [
    "failed_logins", "new_device", "mean_gap_s",
    "geo_change", "night_share", "peer_group",
]


def generate_auth_data(stage="homework"):
    """Synthetic 15-minute authentication windows, seed 42."""
    if stage not in {"homework", "session"}:
        raise ValueError("stage must be homework or session")
    rng = np.random.default_rng(42)
    times = pd.date_range("2026-01-01", periods=30 * 96,
                         freq="15min", tz="UTC")
    window_end = times.repeat(20)
    user_no = np.tile(np.arange(20), len(times))
    service = user_no >= 16
    n = len(window_end)
    day_no = (window_end - times[0]).days.to_numpy() + 1
    threat_rate = np.where(day_no >= 27, 0.015, 0.012)
    target = rng.binomial(1, threat_rate, n)
    legitimate_batch = (rng.random(n) < 0.006) & (target == 0)
    failed = rng.poisson(np.where(service, 2.2, 1.2) + 2 * target)
    failed += legitimate_batch * rng.integers(5, 10, n)
    new_device = rng.binomial(
        1, np.where(target == 1, 0.45, np.where(service, 0.01, 0.08)))
    mean_gap = rng.gamma(2.0, 12.0, n)
    mean_gap[target == 1] = rng.gamma(2.0, 4.5, int(target.sum()))
    mean_gap[legitimate_batch] = rng.gamma(
        2.0, 1.0, int(legitimate_batch.sum()))
    geo_change = rng.binomial(
        1, np.where(target == 1, 0.25, np.where(service, 0.005, 0.02)))
    night_share = rng.beta(1.0, 5.0, n)
    night_share[service] = rng.beta(3.0, 3.0, int(service.sum()))
    night_share[target == 1] = rng.beta(2.5, 3.5, int(target.sum()))
    # Late-period ordinary service activity changes slightly.
    late_service = (day_no >= 27) & service & (target == 0)
    failed[late_service] += rng.binomial(1, 0.5, int(late_service.sum()))
    mean_gap[rng.random(n) < 0.01] = np.nan
    data = pd.DataFrame({
        "row_id": [f"AUTH-{i:06d}" for i in range(n)],
        "window_end": window_end,
        "user_id": [f"u{i:02d}" for i in user_no],
        "peer_group": np.where(service, "service", "office"),
        "failed_logins": failed,
        "new_device": new_device,
        "mean_gap_s": mean_gap,
        "geo_change": geo_change,
        "night_share": night_share,
        "target": target,
    })
    data["label_ready_at"] = data["window_end"] + pd.Timedelta(days=1)
    if stage == "homework":
        data = data[data["window_end"] < pd.Timestamp("2026-01-27", tz="UTC")]
    return data.reset_index(drop=True)

In [ ]:
data = generate_auth_data(stage="homework")

print("Размер данных:", data.shape)
print("\nПервые 5 строк:")
display(data.head())

Размер данных: (49920, 11)

Первые 5 строк:


,row_id,window_end,user_id,peer_group,failed_logins,new_device,mean_gap_s,geo_change,night_share,target,label_ready_at
0,AUTH-000000,2026-01-01 00:00:00+00:00,u00,office,1,0,16.722422,0,0.071058,0,2026-01-02 00:00:00+00:00
1,AUTH-000001,2026-01-01 00:00:00+00:00,u01,office,1,0,13.049162,0,0.054270,0,2026-01-02 00:00:00+00:00
2,AUTH-000002,2026-01-01 00:00:00+00:00,u02,office,3,0,7.350477,0,0.147130,0,2026-01-02 00:00:00+00:00
3,AUTH-000003,2026-01-01 00:00:00+00:00,u03,office,0,0,36.073570,0,0.360504,0,2026-01-02 00:00:00+00:00
4,AUTH-000004,2026-01-01 00:00:00+00:00,u04,office,2,0,63.066878,0,0.144373,0,2026-01-02 00:00:00+00:00


# ML-конвейер для триажа событий информационной безопасности

## 1. Постановка задачи

Цель работы — разработать воспроизводимый конвейер машинного обучения для приоритизации событий аутентификации в системе мониторинга информационной безопасности (SOC).

Система анализирует 15-минутные окна аутентификации и формирует ранжированную очередь событий, которые требуют внимания аналитика.

Основное эксплуатационное ограничение — не более восьми проверок за один календарный день по UTC для всех 20 учебных учётных записей. Поэтому задача заключается не в автоматической блокировке подозрительных пользователей, а в выборе наиболее приоритетных событий при ограниченных ресурсах аналитика.

В работе сравниваются три подхода: Logistic Regression, Random Forest и Isolation Forest. Первые два используются для классификации событий, третий — для обнаружения аномалий.

**Принцип работы:** модель оценивает подозрительность событий, после чего конвейер выбирает восемь событий с наибольшими оценками за каждый UTC-день. Итоговые решения о необходимости расследования принимает человек.

Целевая переменная `target` используется для обучения и последующей оценки моделей. Значение `target = 1` обозначает событие, требующее расследования согласно сценарию генерации данных.

## 2. Паспорт и аудит данных

### 2.1. Версии программного окружения

In [ ]:
import sys
import numpy as np
import pandas as pd
import sklearn
import scipy
import joblib
import threadpoolctl

print("Python:", sys.version)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("SciPy:", scipy.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Joblib:", joblib.__version__)
print("Threadpoolctl:", threadpoolctl.__version__)

Python: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
NumPy: 2.4.6
Pandas: 2.2.3
SciPy: 1.16.3
Scikit-learn: 1.6.1
Joblib: 1.6.0
Threadpoolctl: 3.7.0


### 2.2. Разделение данных по времени

In [ ]:
# Границы временных интервалов (UTC)
train_start = pd.Timestamp("2026-01-01", tz="UTC")
train_end = pd.Timestamp("2026-01-15", tz="UTC")

valid_start = pd.Timestamp("2026-01-16", tz="UTC")
valid_end = pd.Timestamp("2026-01-21", tz="UTC")

test_start = pd.Timestamp("2026-01-22", tz="UTC")
test_end = pd.Timestamp("2026-01-27", tz="UTC")

# Разделение данных
train = data[
    (data["window_end"] >= train_start) &
    (data["window_end"] < train_end)
].copy()

valid = data[
    (data["window_end"] >= valid_start) &
    (data["window_end"] < valid_end)
].copy()

test = data[
    (data["window_end"] >= test_start) &
    (data["window_end"] < test_end)
].copy()

# Вывод размеров частей
print("TRAIN")
print("Количество строк:", len(train))
print("Начало:", train["window_end"].min())
print("Конец:", train["window_end"].max())

print("\nVALID")
print("Количество строк:", len(valid))
print("Начало:", valid["window_end"].min())
print("Конец:", valid["window_end"].max())

print("\nTEST")
print("Количество строк:", len(test))
print("Начало:", test["window_end"].min())
print("Конец:", test["window_end"].max())

TRAIN
Количество строк: 26880
Начало: 2026-01-01 00:00:00+00:00
Конец: 2026-01-14 23:45:00+00:00

VALID
Количество строк: 9600
Начало: 2026-01-16 00:00:00+00:00
Конец: 2026-01-20 23:45:00+00:00

TEST
Количество строк: 9600
Начало: 2026-01-22 00:00:00+00:00
Конец: 2026-01-26 23:45:00+00:00


### 2.3. Проверка временного протокола и доступности меток

In [ ]:
# Проверяем доступность меток
print("Последнее время готовности метки TRAIN:")
print(train["label_ready_at"].max())

print("\nНачало VALID:")
print(valid_start)

print("\nПоследнее время готовности метки VALID:")
print(valid["label_ready_at"].max())

print("\nНачало TEST:")
print(test_start)

# Проверяем доступность меток
assert train["label_ready_at"].max() <= valid_start
assert valid["label_ready_at"].max() <= test_start

# Проверяем отсутствие пересечения row_id
assert set(train["row_id"]).isdisjoint(set(valid["row_id"]))
assert set(train["row_id"]).isdisjoint(set(test["row_id"]))
assert set(valid["row_id"]).isdisjoint(set(test["row_id"]))

print("Все проверки временного протокола пройдены!")
print("Пересечений row_id нет.")
print("Метки доступны к началу соответствующих проверочных периодов.")

Последнее время готовности метки TRAIN:
2026-01-15 23:45:00+00:00

Начало VALID:
2026-01-16 00:00:00+00:00

Последнее время готовности метки VALID:
2026-01-21 23:45:00+00:00

Начало TEST:
2026-01-22 00:00:00+00:00
Все проверки временного протокола пройдены!
Пересечений row_id нет.
Метки доступны к началу соответствующих проверочных периодов.


### 2.4. Аудит обучающей и валидационной выборок

In [ ]:
# Разрешённые признаки
FEATURES = [
    "failed_logins",
    "new_device",
    "mean_gap_s",
    "geo_change",
    "night_share",
    "peer_group"
]

# Числовые и категориальные признаки
NUMERIC_FEATURES = [
    "failed_logins",
    "new_device",
    "mean_gap_s",
    "geo_change",
    "night_share"
]

CATEGORICAL_FEATURES = ["peer_group"]

# Функция для проверки train и valid
def audit_dataset(df, name):
    print("=" * 50)
    print("АУДИТ:", name)
    print("=" * 50)

    print("\nРазмер:")
    print(df.shape)

    print("\nТипы данных:")
    print(df.dtypes)

    print("\nПропуски:")
    print(df[FEATURES].isna().sum())

    print("\nКоличество каждого класса:")
    print(df["target"].value_counts())

    print("\nДоли классов:")
    print(df["target"].value_counts(normalize=True))

    print("\nСостав групп:")
    print(df["peer_group"].value_counts())

    print("\nУникальные row_id:")
    print(df["row_id"].nunique())

    print("\nУникальные user_id:")
    print(df["user_id"].nunique())

    print("\nДиапазоны числовых признаков:")
    print(df[NUMERIC_FEATURES].describe())

# Выполняем аудит
audit_dataset(train, "TRAIN")
audit_dataset(valid, "VALID")

АУДИТ: TRAIN

Размер:
(26880, 11)

Типы данных:
row_id                         object
window_end        datetime64[ns, UTC]
user_id                        object
peer_group                     object
failed_logins                   int64
new_device                      int64
mean_gap_s                    float64
geo_change                      int64
night_share                   float64
target                          int64
label_ready_at    datetime64[ns, UTC]
dtype: object

Пропуски:
failed_logins      0
new_device         0
mean_gap_s       299
geo_change         0
night_share        0
peer_group         0
dtype: int64

Количество каждого класса:
target
0    26533
1      347
Name: count, dtype: int64

Доли классов:
target
0    0.987091
1    0.012909
Name: proportion, dtype: float64

Состав групп:
peer_group
office     21504
service     5376
Name: count, dtype: int64

Уникальные row_id:
26880

Уникальные user_id:
20

Диапазоны числовых признаков:
       failed_logins    new_device   

 mean_gap_s обнаружены пропущенные значения: 299 в train и 98 в valid, дисбаланс классов: 1,29% в train и 1,10% в valid -> precision@8 и recall@8

In [ ]:
# Признаки для обучения моделей
NUMERIC_FEATURES = [
    "failed_logins",
    "new_device",
    "mean_gap_s",
    "geo_change",
    "night_share"
]

CATEGORICAL_FEATURES = [
    "peer_group"
]

FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

TARGET = "target"

# Матрицы признаков и целевую переменную
X_train = train[FEATURES].copy()
y_train = train[TARGET].copy()

X_valid = valid[FEATURES].copy()
y_valid = valid[TARGET].copy()

print("Train:", X_train.shape, y_train.shape)
print("Valid:", X_valid.shape, y_valid.shape)

print("\nПропуски в train:")
print(X_train.isna().sum())

print("\nРаспределение классов:")
print(y_train.value_counts())

Train: (26880, 6) (26880,)
Valid: (9600, 6) (9600,)

Пропуски в train:
failed_logins      0
new_device         0
mean_gap_s       299
geo_change         0
night_share        0
peer_group         0
dtype: int64

Распределение классов:
target
0    26533
1      347
Name: count, dtype: int64


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Предобработка числовых признаков для Logistic Regression
numeric_lr = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Предобработка категориального признака для Logistic Regression
categorical_lr = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Общий препроцессор для Logistic Regression
preprocessor_lr = ColumnTransformer(transformers=[
    ("numeric", numeric_lr, NUMERIC_FEATURES),
    ("categorical", categorical_lr, CATEGORICAL_FEATURES)
])


# Предобработка числовых признаков для моделей на деревьях
numeric_tree = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

# Предобработка категориального признака для моделей на деревьях
categorical_tree = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Общий препроцессор для Random Forest и Isolation Forest
preprocessor_tree = ColumnTransformer(transformers=[
    ("numeric", numeric_tree, NUMERIC_FEATURES),
    ("categorical", categorical_tree, CATEGORICAL_FEATURES)
])

print("Препроцессоры успешно созданы.")
print("Logistic Regression: заполнение пропусков + масштабирование + кодирование.")
print("Tree-based models: заполнение пропусков + кодирование.")

Препроцессоры успешно созданы.
Logistic Regression: заполнение пропусков + масштабирование + кодирование.
Tree-based models: заполнение пропусков + кодирование.


In [21]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

#LR
lr_model = Pipeline(steps=[
    ("preprocessor", preprocessor_lr),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

#RF
rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor_tree),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        max_features="sqrt",
        min_samples_leaf=5,
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1
    ))
])

#Обучение
lr_model.fit(X_train, y_train)
rf_model.fit(X_train, y_train)

print("Обе модели успешно обучены.")

Обе модели успешно обучены.


In [22]:
# Вероятности положительного класса (target = 1)
lr_scores = lr_model.predict_proba(X_valid)[:, 1]
rf_scores = rf_model.predict_proba(X_valid)[:, 1]

# Сохраняем результаты
valid_scores = valid[
    ["row_id", "window_end", "user_id", "target"]
].copy()

valid_scores["lr_score"] = lr_scores
valid_scores["rf_score"] = rf_scores

print("Первые 10 результатов:")
display(valid_scores.head(10))

Первые 10 результатов:


,row_id,window_end,user_id,target,lr_score,rf_score
28800,AUTH-028800,2026-01-16 00:00:00+00:00,u00,0,0.419656,0.036032
28801,AUTH-028801,2026-01-16 00:00:00+00:00,u01,0,0.068790,0.000000
28802,AUTH-028802,2026-01-16 00:00:00+00:00,u02,0,0.060227,0.028640
28803,AUTH-028803,2026-01-16 00:00:00+00:00,u03,0,0.029530,0.000619
28804,AUTH-028804,2026-01-16 00:00:00+00:00,u04,0,0.060947,0.000000
28805,AUTH-028805,2026-01-16 00:00:00+00:00,u05,0,0.159463,0.000000
28806,AUTH-028806,2026-01-16 00:00:00+00:00,u06,0,0.373089,0.227030
28807,AUTH-028807,2026-01-16 00:00:00+00:00,u07,0,0.011045,0.001910
28808,AUTH-028808,2026-01-16 00:00:00+00:00,u08,0,0.694876,0.334741
28809,AUTH-028809,2026-01-16 00:00:00+00:00,u09,0,0.193190,0.010195


Isolation Forrest

In [23]:
from sklearn.ensemble import IsolationForest
from sklearn.pipeline import Pipeline

if_model = Pipeline(steps=[
    ("preprocessor", preprocessor_tree),
    ("detector", IsolationForest(
        n_estimators=300,
        contamination="auto",
        random_state=42,
        n_jobs=-1
    ))
])

if_model.fit(X_train)

print("Isolation Forest: обучение завершено")

Isolation Forest: обучение завершено


###Формирование очереди подозрительных событий

In [31]:
valid_scores["day"] = valid_scores["window_end"].dt.floor("D")
display(valid_scores[["window_end", "day"]].head())

lr_sorted = valid_scores.sort_values(
    by=["day", "lr_score", "row_id"],
    ascending=[True, False, True]
)
display(lr_sorted.head(10))

lr_queue = lr_sorted.groupby("day").head(8)

print("Всего событий:", len(lr_queue))
print("\nКоличество событий по дням:")
print(lr_queue.groupby("day").size())

,window_end,day
28800,2026-01-16 00:00:00+00:00,2026-01-16 00:00:00+00:00
28801,2026-01-16 00:00:00+00:00,2026-01-16 00:00:00+00:00
28802,2026-01-16 00:00:00+00:00,2026-01-16 00:00:00+00:00
28803,2026-01-16 00:00:00+00:00,2026-01-16 00:00:00+00:00
28804,2026-01-16 00:00:00+00:00,2026-01-16 00:00:00+00:00


,row_id,window_end,user_id,target,lr_score,rf_score,day
29007,AUTH-029007,2026-01-16 02:30:00+00:00,u07,1,0.999969,0.997642,2026-01-16 00:00:00+00:00
29544,AUTH-029544,2026-01-16 09:15:00+00:00,u04,1,0.999932,0.996922,2026-01-16 00:00:00+00:00
29461,AUTH-029461,2026-01-16 08:15:00+00:00,u01,1,0.999882,0.978794,2026-01-16 00:00:00+00:00
28951,AUTH-028951,2026-01-16 01:45:00+00:00,u11,0,0.999653,0.193047,2026-01-16 00:00:00+00:00
29037,AUTH-029037,2026-01-16 02:45:00+00:00,u17,0,0.999533,0.126240,2026-01-16 00:00:00+00:00
29621,AUTH-029621,2026-01-16 10:15:00+00:00,u01,0,0.998815,0.694426,2026-01-16 00:00:00+00:00
29988,AUTH-029988,2026-01-16 14:45:00+00:00,u08,1,0.997519,0.966635,2026-01-16 00:00:00+00:00
30684,AUTH-030684,2026-01-16 23:30:00+00:00,u04,0,0.997224,0.081349,2026-01-16 00:00:00+00:00
29391,AUTH-029391,2026-01-16 07:15:00+00:00,u11,1,0.996023,0.982901,2026-01-16 00:00:00+00:00
29847,AUTH-029847,2026-01-16 13:00:00+00:00,u07,0,0.993955,0.183142,2026-01-16 00:00:00+00:00


Всего событий: 40

Количество событий по дням:
day
2026-01-16 00:00:00+00:00    8
2026-01-17 00:00:00+00:00    8
2026-01-18 00:00:00+00:00    8
2026-01-19 00:00:00+00:00    8
2026-01-20 00:00:00+00:00    8
dtype: int64


In [33]:
rf_sorted = valid_scores.sort_values(
    by=["day", "rf_score", "row_id"],
    ascending=[True, False, True]
)

display(rf_sorted.head(10))

rf_queue = rf_sorted.groupby("day").head(8)

print("Всего событий:", len(rf_queue))
print("\nКоличество событий по дням:")
print(rf_queue.groupby("day").size())

,row_id,window_end,user_id,target,lr_score,rf_score,day
29007,AUTH-029007,2026-01-16 02:30:00+00:00,u07,1,0.999969,0.997642,2026-01-16 00:00:00+00:00
29544,AUTH-029544,2026-01-16 09:15:00+00:00,u04,1,0.999932,0.996922,2026-01-16 00:00:00+00:00
29391,AUTH-029391,2026-01-16 07:15:00+00:00,u11,1,0.996023,0.982901,2026-01-16 00:00:00+00:00
29435,AUTH-029435,2026-01-16 07:45:00+00:00,u15,1,0.992666,0.980017,2026-01-16 00:00:00+00:00
29461,AUTH-029461,2026-01-16 08:15:00+00:00,u01,1,0.999882,0.978794,2026-01-16 00:00:00+00:00
29988,AUTH-029988,2026-01-16 14:45:00+00:00,u08,1,0.997519,0.966635,2026-01-16 00:00:00+00:00
29221,AUTH-029221,2026-01-16 05:15:00+00:00,u01,1,0.986815,0.940119,2026-01-16 00:00:00+00:00
29589,AUTH-029589,2026-01-16 09:45:00+00:00,u09,0,0.973677,0.911890,2026-01-16 00:00:00+00:00
30288,AUTH-030288,2026-01-16 18:30:00+00:00,u08,0,0.961821,0.893435,2026-01-16 00:00:00+00:00
30584,AUTH-030584,2026-01-16 22:15:00+00:00,u04,1,0.993654,0.881806,2026-01-16 00:00:00+00:00


Всего событий: 40

Количество событий по дням:
day
2026-01-16 00:00:00+00:00    8
2026-01-17 00:00:00+00:00    8
2026-01-18 00:00:00+00:00    8
2026-01-19 00:00:00+00:00    8
2026-01-20 00:00:00+00:00    8
dtype: int64


In [37]:
if_scores = -if_model.score_samples(X_valid)

valid_scores["if_score"] = if_scores

print("Столбец if_score добавлен.")
display(valid_scores.head())

Столбец if_score добавлен.


,row_id,window_end,user_id,target,lr_score,rf_score,day,if_score
28800,AUTH-028800,2026-01-16 00:00:00+00:00,u00,0,0.419656,0.036032,2026-01-16 00:00:00+00:00,0.411204
28801,AUTH-028801,2026-01-16 00:00:00+00:00,u01,0,0.068790,0.000000,2026-01-16 00:00:00+00:00,0.397140
28802,AUTH-028802,2026-01-16 00:00:00+00:00,u02,0,0.060227,0.028640,2026-01-16 00:00:00+00:00,0.381255
28803,AUTH-028803,2026-01-16 00:00:00+00:00,u03,0,0.029530,0.000619,2026-01-16 00:00:00+00:00,0.404006
28804,AUTH-028804,2026-01-16 00:00:00+00:00,u04,0,0.060947,0.000000,2026-01-16 00:00:00+00:00,0.374558


In [38]:
if_sorted = valid_scores.sort_values(
    by=["day", "if_score", "row_id"],
    ascending=[True, False, True]
)

if_queue = if_sorted.groupby("day").head(8)

print("Всего событий:", len(if_queue))
print("\nКоличество событий по дням:")
print(if_queue.groupby("day").size())

Всего событий: 40

Количество событий по дням:
day
2026-01-16 00:00:00+00:00    8
2026-01-17 00:00:00+00:00    8
2026-01-18 00:00:00+00:00    8
2026-01-19 00:00:00+00:00    8
2026-01-20 00:00:00+00:00    8
dtype: int64


In [40]:
print("\nВсего положительных событий в validation:")
print(valid["target"].sum())


Всего положительных событий в validation:
106


In [42]:
def evaluate_queue(queue, total_positives):
    tp = int((queue["target"] == 1).sum())
    fp = int((queue["target"] == 0).sum())
    selected = len(queue)

    precision = tp / selected if selected else 0
    recall = tp / total_positives if total_positives else 0

    max_daily = int(queue.groupby("day").size().max())

    return {
        "selected": selected,
        "max_per_day": max_daily,
        "TP": tp,
        "FP": fp,
        "Precision@8": precision,
        "Recall@8": recall
    }
total_positives = int(valid["target"].sum())

results = pd.DataFrame({
    "Logistic Regression": evaluate_queue(lr_queue, total_positives),
    "Random Forest": evaluate_queue(rf_queue, total_positives),
    "Isolation Forest": evaluate_queue(if_queue, total_positives)
}).T

display(results.round(3))

,selected,max_per_day,TP,FP,Precision@8,Recall@8
Logistic Regression,40.0,8.0,16.0,24.0,0.400,0.151
Random Forest,40.0,8.0,32.0,8.0,0.800,0.302
Isolation Forest,40.0,8.0,13.0,27.0,0.325,0.123


####Проверка

In [43]:
thresholds = [0.1, 0.3, 0.5]

threshold_results = []

for threshold in thresholds:

    selected = valid_scores[
        valid_scores["rf_score"] >= threshold
    ]

    tp = int((selected["target"] == 1).sum())
    fp = int((selected["target"] == 0).sum())

    total_selected = len(selected)

    daily_counts = (
        selected.groupby("day")
        .size()
        .reindex(
            valid_scores["day"].drop_duplicates().sort_values(),
            fill_value=0
        )
    )

    threshold_results.append({
        "threshold": threshold,
        "selected": total_selected,
        "TP": tp,
        "FP": fp,
        "Precision": tp / total_selected if total_selected else 0,
        "Recall": tp / total_positives,
        "max_per_day": int(daily_counts.max()),
        "days_over_budget": int((daily_counts > 8).sum())
    })

threshold_results = pd.DataFrame(threshold_results)

display(threshold_results.round(3))

,threshold,selected,TP,FP,Precision,Recall,max_per_day,days_over_budget
0,0.1,1109,92,1017,0.083,0.868,251,5
1,0.3,371,78,293,0.210,0.736,86,5
2,0.5,171,58,113,0.339,0.547,40,5


### Фиксация модели перед тестированием

- Выбранная модель: Random Forest.
- `n_estimators=300`
- `max_features="sqrt"`
- `min_samples_leaf=5`
- `class_weight="balanced_subsample"`
- `random_state=42`
- Признаки: `failed_logins`, `new_device`, `mean_gap_s`, `geo_change`, `night_share`, `peer_group`.
- Правило формирования очереди: 8 событий с наибольшим score за каждый UTC-день.
- Выбор модели выполнен по результатам validation.
- Test не использовался для выбора модели и параметров.

In [45]:
X_test = test[FEATURES].copy()

# Оценки Random Forest
test_scores = test[
    ["row_id", "window_end", "user_id"]
].copy()

test_scores["rf_score"] = rf_model.predict_proba(X_test)[:, 1]
test_scores["day"] = test_scores["window_end"].dt.floor("D")
test_sorted = test_scores.sort_values(
    by=["day", "rf_score", "row_id"],
    ascending=[True, False, True]
)
test_queue = test_sorted.groupby("day").head(8)

print("Количество событий в очереди:", len(test_queue))
print("\nСобытий по дням:")
print(test_queue.groupby("day").size())

Количество событий в очереди: 40

Событий по дням:
day
2026-01-22 00:00:00+00:00    8
2026-01-23 00:00:00+00:00    8
2026-01-24 00:00:00+00:00    8
2026-01-25 00:00:00+00:00    8
2026-01-26 00:00:00+00:00    8
dtype: int64


##Test

In [46]:
# Добавляем истинные метки только после формирования очереди
test_evaluated = test_queue.merge(
    test[["row_id", "target"]],
    on="row_id",
    how="left"
)

# Подсчет результаты
tp = int((test_evaluated["target"] == 1).sum())
fp = int((test_evaluated["target"] == 0).sum())
total_selected = len(test_evaluated)
total_positives_test = int(test["target"].sum())

precision_test = tp / total_selected
recall_test = tp / total_positives_test

test_results = pd.DataFrame([{
    "selected": total_selected,
    "max_per_day": int(test_evaluated.groupby("day").size().max()),
    "TP": tp,
    "FP": fp,
    "total_positives": total_positives_test,
    "Precision@8": precision_test,
    "Recall@8": recall_test
}])

display(test_results.round(3))

,selected,max_per_day,TP,FP,total_positives,Precision@8,Recall@8
0,40,8,28,12,106,0.7,0.264


Ошибки

In [48]:
test_audit = test_scores.merge(
    test[["row_id", "target"] + FEATURES],
    on="row_id",
    how="left"
)
selected_ids = set(test_queue["row_id"])
test_audit["selected"] = test_audit["row_id"].isin(selected_ids)

false_positives = test_audit[
    (test_audit["selected"]) & (test_audit["target"] == 0)
]

false_negatives = test_audit[
    (~test_audit["selected"]) & (test_audit["target"] == 1)
]

print("Ложные срабатывания (FP):", len(false_positives))
print("Пропущенные подозрительные события (FN):", len(false_negatives))

print("\nПримеры ложных срабатываний:")
display(false_positives.head(5))

print("\nПримеры пропущенных событий:")
display(false_negatives.head(5))

Ложные срабатывания (FP): 12
Пропущенные подозрительные события (FN): 78

Примеры ложных срабатываний:


,row_id,window_end,user_id,rf_score,day,target,failed_logins,new_device,mean_gap_s,geo_change,night_share,peer_group,selected
158,AUTH-040478,2026-01-22 01:45:00+00:00,u18,0.758046,2026-01-22 00:00:00+00:00,0,3,1,10.659000,0,0.519802,service,True
649,AUTH-040969,2026-01-22 08:00:00+00:00,u09,0.834196,2026-01-22 00:00:00+00:00,0,4,1,18.357488,0,0.294852,office,True
685,AUTH-041005,2026-01-22 08:30:00+00:00,u05,0.752031,2026-01-22 00:00:00+00:00,0,2,0,6.610552,0,0.790634,office,True
2150,AUTH-042470,2026-01-23 02:45:00+00:00,u10,0.976600,2026-01-23 00:00:00+00:00,0,4,1,6.850384,0,0.480933,office,True
2603,AUTH-042923,2026-01-23 08:30:00+00:00,u03,0.822482,2026-01-23 00:00:00+00:00,0,6,1,2.541629,0,0.179221,office,True



Примеры пропущенных событий:


,row_id,window_end,user_id,rf_score,day,target,failed_logins,new_device,mean_gap_s,geo_change,night_share,peer_group,selected
208,AUTH-040528,2026-01-22 02:30:00+00:00,u08,0.002218,2026-01-22 00:00:00+00:00,1,0,0,14.809038,0,0.197474,office,False
294,AUTH-040614,2026-01-22 03:30:00+00:00,u14,0.246532,2026-01-22 00:00:00+00:00,1,5,1,17.555758,0,0.105670,office,False
310,AUTH-040630,2026-01-22 03:45:00+00:00,u10,0.060769,2026-01-22 00:00:00+00:00,1,2,0,13.839675,0,0.324437,office,False
566,AUTH-040886,2026-01-22 07:00:00+00:00,u06,0.577266,2026-01-22 00:00:00+00:00,1,2,1,30.386968,1,0.654951,office,False
659,AUTH-040979,2026-01-22 08:00:00+00:00,u19,0.285077,2026-01-22 00:00:00+00:00,1,2,0,4.357037,0,0.218383,service,False


In [59]:
test_audit["error_type"] = "Other"

test_audit.loc[
    (test_audit["selected"] == True) &
    (test_audit["target"] == 0),
    "error_type"
] = "FP"

test_audit.loc[
    (test_audit["selected"] == False) &
    (test_audit["target"] == 1),
    "error_type"
] = "FN"

print("Средние значения FP и FN")

error_summary = (
    test_audit[
        test_audit["error_type"].isin(["FP", "FN"])
    ]
    .groupby("error_type")[NUMERIC_FEATURES]
    .mean()
)

display(error_summary.round(3))

print("Распределение ошибок по группам пользователей")

group_summary = (
    test_audit[
        test_audit["error_type"].isin(["FP", "FN"])
    ]
    .groupby(["error_type", "peer_group"])
    .size()
    .unstack(fill_value=0)
)

display(group_summary)

test_audit["result"] = "TN"

test_audit.loc[
    (test_audit["selected"] == True) &
    (test_audit["target"] == 1),
    "result"
] = "TP"

test_audit.loc[
    (test_audit["selected"] == True) &
    (test_audit["target"] == 0),
    "result"
] = "FP"

test_audit.loc[
    (test_audit["selected"] == False) &
    (test_audit["target"] == 1),
    "result"
] = "FN"

print("Распределение результатов")

display(
    test_audit["result"]
    .value_counts()
    .reindex(["TP", "FP", "TN", "FN"], fill_value=0)
    .to_frame("Количество")
)

Средние значения FP и FN


,failed_logins,new_device,mean_gap_s,geo_change,night_share
error_type,,,,,
FN,2.846,0.397,10.123,0.205,0.371
FP,3.083,0.917,9.290,0.000,0.497


Распределение ошибок по группам пользователей


peer_group,office,service
error_type,,
FN,64,14
FP,10,2


Распределение результатов


,Количество
result,
TP,28
FP,12
TN,9482
FN,78


In [61]:
print("Размер test:", test.shape)
print("Уникальные row_id:", test["row_id"].nunique())
print("Уникальные user_id:", test["user_id"].nunique())

print("Период:", test["window_end"].min(), "—", test["window_end"].max())

print("\nПропуски в признаках:")
display(test[FEATURES].isna().sum().to_frame("missing"))

print("\nПересечения row_id:")
print("train/test:", len(set(train["row_id"]) & set(test["row_id"])))
print("valid/test:", len(set(valid["row_id"]) & set(test["row_id"])))

Размер test: (9600, 11)
Уникальные row_id: 9600
Уникальные user_id: 20
Период: 2026-01-22 00:00:00+00:00 — 2026-01-26 23:45:00+00:00

Пропуски в признаках:


,missing
failed_logins,0
new_device,0
mean_gap_s,109
geo_change,0
night_share,0
peer_group,0



Пересечения row_id:
train/test: 0
valid/test: 0


In [60]:
group_errors = []

for group in test_audit["peer_group"].unique():

    subset = test_audit[test_audit["peer_group"] == group]

    # Все подозрительные события в группе
    positives = subset[subset["target"] == 1]

    # Все обычные события в группе
    negatives = subset[subset["target"] == 0]

    # Пропущенные подозрительные события
    fn = ((positives["selected"] == False)).sum()

    # Ложные срабатывания
    fp = ((negatives["selected"] == True)).sum()

    group_errors.append({
        "peer_group": group,
        "total_positive": len(positives),
        "FN": fn,
        "FN_rate": fn / len(positives) if len(positives) else 0,
        "total_negative": len(negatives),
        "FP": fp,
        "FP_rate": fp / len(negatives) if len(negatives) else 0
    })

group_errors = pd.DataFrame(group_errors)

display(group_errors.round(3))

,peer_group,total_positive,FN,FN_rate,total_negative,FP,FP_rate
0,office,89,64,0.719,7591,10,0.001
1,service,17,14,0.824,1903,2,0.001


Анализ ошибок показал, что Random Forest чаще пропускает подозрительные события, чем ошибочно выбирает обычные. На тестовой выборке выявлено 78 FN и 12 FP. Средняя доля новых устройств среди FP составила 0.917, что указывает на возможную избыточную чувствительность модели к этому признаку. При этом часть FN имеет признаки смены геолокации, но не попадает в очередь. Доля пропусков составила 71.9% для office и 82.4% для service. Из-за небольшого числа положительных примеров в группе service этот результат требует дополнительной проверки.

##  Заключение и рекомендация для SOC

По результатам эксперимента для формирования очереди событий рекомендуется использовать модель Random Forest. Она показала лучшие результаты на валидационной выборке среди трёх рассмотренных подходов: Logistic Regression, Random Forest и Isolation Forest. Precision@8 составила 0.800, Recall@8 — 0.302.

На тестовой выборке модель отобрала 40 событий за пять дней — по восемь событий на каждый UTC-день. Из них 28 оказались истинно положительными, 12 — ложноположительными. Precision@8 составила 0.700, Recall@8 — 0.264. Таким образом, модель выполняет задачу приоритизации событий в условиях ограниченного ресурса аналитиков, однако пропускает значительную часть событий, требующих расследования.

Анализ ошибок выявил 78 ложноотрицательных и 12 ложноположительных результатов. Среди ложноположительных событий в 91,7% случаев наблюдалось использование нового устройства. Это может указывать на избыточную чувствительность модели к данному признаку. Доля пропусков составила 71,9% для группы office и 82,4% для service. Последний показатель требует дополнительного исследования, поскольку количество положительных примеров в этой группе невелико.

Эксперимент с порогами вероятности 0.1, 0.3 и 0.5 показал, что простое применение порога не обеспечивает соблюдение ограничения в восемь проверок в день. Поэтому рекомендуется сохранить правило ежедневного отбора Top-8 с сортировкой по оценке подозрительности и дополнительной сортировкой по `row_id` для разрешения равенства оценок.

Перед внедрением необходимо провести дополнительные проверки:

1. Проверить модель на обезличенных реальных журналах аутентификации с подтверждёнными результатами расследований.
2. Запустить модель в теневом режиме и сравнить её рекомендации с решениями аналитиков SOC.
3. Оценить стабильность качества модели во времени, проверить изменение распределения признаков и частоту пропусков.
4. Проверить журналирование, разграничение доступа и сохранение версий модели и результатов отбора.

Полученные результаты ограничены синтетическим набором данных, 20 учебными учётными записями и коротким временным периодом. Поэтому их нельзя считать доказательством эффективности модели в реальной инфраструктуре.

**Итоговая рекомендация:** использовать Random Forest как вспомогательный инструмент ранжирования событий в SOC с ограничением Top-8 в день. Модель не должна автоматически блокировать учётные записи или заменять решение аналитика. Все отобранные события должны проходить проверку человеком.